In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import datetime

# === GPU Setup ===
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        for device in physical_devices:
            tf.config.experimental.set_memory_growth(device, True)
        print("[INFO] GPU memory growth enabled.")
    except Exception as e:
        print(f"[ERROR] Could not set memory growth: {e}")
else:
    print("[INFO] No GPU found, running on CPU.")

# === Load Data ===
print("[INFO] Loading dataset...")
X = np.load("X.npy").astype(np.float32) / 255.0
y = np.load("y.npy")

with open("label_dict.json") as f:
    label_dict = json.load(f)
idx_to_label = {v: k for k, v in label_dict.items()}

# Handle mismatch in y labels and label_dict
unique_classes = np.unique(y)
num_classes = len(unique_classes)
actual_labels = [idx_to_label[str(cls)] if str(cls) in idx_to_label else f"Class_{cls}" for cls in unique_classes]

# === Train-test split ===
print("[INFO] Splitting dataset...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === I3D-like model definition ===
def build_i3d_like_model(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv3D(64, (3, 7, 7), strides=(1, 2, 2), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same')(x)

    x = layers.Conv3D(128, (3, 3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling3D((2, 2, 2), padding='same')(x)

    x = layers.Conv3D(256, (3, 3, 3), padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling3D()(x)

    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs)

# === Build & Compile the Model ===
input_shape = X_train.shape[1:]  # (frames, height, width, channels)
print(f"[INFO] Building model with input shape: {input_shape}, num classes: {num_classes}")
model = build_i3d_like_model(input_shape, num_classes)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# === Callbacks ===
log_dir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_cb = callbacks.TensorBoard(log_dir=log_dir)
checkpoint_cb = callbacks.ModelCheckpoint("best_model.h5", save_best_only=True, monitor='val_accuracy', mode='max')
earlystop_cb = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# === Train Model ===
print("[INFO] Training model...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=4,
    callbacks=[tensorboard_cb, checkpoint_cb, earlystop_cb]
)

# Save final model
model.save("final_i3d_model.h5")
print("[INFO] Final model saved.")

# === Evaluation ===
print("[INFO] Evaluating model...")
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Classification report
print("\n[INFO] Classification Report:")
print(classification_report(y_test, y_pred, target_names=actual_labels))

# Confusion matrix
plt.figure(figsize=(10, 7))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d',
            xticklabels=actual_labels, yticklabels=actual_labels, cmap='Blues')
plt.title("Confusion Matrix - I3D-like Model")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig("confusion_matrix.png")
plt.show()


[INFO] No GPU found, running on CPU.
[INFO] Loading dataset...
[INFO] Splitting dataset...
[INFO] Building model with input shape: (100, 128, 128, 3), num classes: 9


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 100, 128, 128,  │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_6 (Conv3D)               │ (None, 100, 64, 64,    │        28,288 │
│                                 │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_4 (MaxPooling3D)  │ (None, 100, 32, 32,    │             0 │
│                                 │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_7 (Conv3D)               │ (None, 100, 32, 32,    │       221,312 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_5 (MaxPooling3D)  │ (None, 50, 16, 16,     │             0 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_8 (Conv3D)               │ (None, 50, 16, 16,     │       884,992 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling3d_2      │ (None, 256)            │             0 │
│ (GlobalAveragePooling3D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 9)              │         2,313 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,136,905 (4.34 MB)

 Trainable params: 1,136,905 (4.34 MB)

 Non-trainable params: 0 (0.00 B)

[INFO] Training model...
Epoch 1/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.1327 - loss: 2.2239

23/23 ━━━━━━━━━━━━━━━━━━━━ 105s 4s/step - accuracy: 0.1355 - loss: 2.2215 - val_accuracy: 0.3043 - val_loss: 2.2125
Epoch 2/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 102s 4s/step - accuracy: 0.2831 - loss: 2.2328 - val_accuracy: 0.3043 - val_loss: 2.1765
Epoch 3/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 106s 5s/step - accuracy: 0.2389 - loss: 2.1807 - val_accuracy: 0.3043 - val_loss: 2.0975
Epoch 4/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 99s 4s/step - accuracy: 0.2426 - loss: 2.1481 - val_accuracy: 0.3043 - val_loss: 2.0587
Epoch 5/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 99s 4s/step - accuracy: 0.2555 - loss: 2.1382 - val_accuracy: 0.3043 - val_loss: 2.0751
Epoch 6/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 100s 4s/step - accuracy: 0.2929 - loss: 2.0619 - val_accuracy: 0.3043 - val_loss: 2.0561
Epoch 7/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 99s 4s/step - accuracy: 0.2435 - loss: 2.1036 - val_accuracy: 0.3043 - val_loss: 2.0451
Epoch 8/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 100s 4s/step - accuracy: 0.2974 - loss: 2.0841 - val_accuracy: 0.3043 - val_loss: 2.046

[INFO] Final model saved.
[INFO] Evaluating model...
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step

[INFO] Classification Report:


ValueError: Number of classes, 8, does not match size of target_names, 9. Try specifying the labels parameter